# ROMAN + MiniRocket on HandMovementDirection

This notebook shows a minimal end-to-end ROMAN workflow on the UEA `HandMovementDirection` dataset.

It compares:

- baseline MiniRocket on the original input
- MiniRocket on the ROMAN-transformed input

In the paper, ROMAN is presented as a deterministic representation operator: `S=1` is the identity case, while larger values of `S` route temporal scale and coarse temporal position into pseudochannels. For this dataset, the appendix results indicate that MiniRocket improves from approximately `0.468` at `S=1` to `0.565` at `S=3` on average across seeds. This notebook runs one local training example, so your exact accuracy may differ, but it reproduces the intended usage pattern and the channel-expansion behavior.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Helpful defaults for constrained notebook environments.
os.environ.setdefault("NUMBA_CACHE_DIR", os.path.join(tempfile.gettempdir(), "numba_cache"))
os.environ.setdefault("NUMBA_THREADING_LAYER", "workqueue")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))

# Make the local package importable when the notebook is run from this repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    src_dir = candidate / "src"
    if (src_dir / "roman" / "__init__.py").exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError(
        "Could not locate the local 'src/roman' package. "
        "Run the notebook from inside the ROMAN repository or install the package first."
    )

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score
from sktime.datasets import load_UCR_UEA_dataset
from sktime.datatypes import convert_to
from sktime.transformations.panel.padder import PaddingTransformer
from sktime.transformations.panel.rocket import MiniRocketMultivariate

from roman import RomanOperator

In [43]:
DATASET_NAME = "EthanolConcentration"# "StandWalkJump" "EthanolConcentration" "SelfRegulationSCP2"
NUM_KERNELS = 10000
RANDOM_STATE = 1

def max_series_length(X):
    max_len = 0
    for col in X.columns:
        for idx in X.index:
            seq = X.loc[idx, col]
            if hasattr(seq, "shape"):
                max_len = max(max_len, seq.shape[0])
            else:
                max_len = max(max_len, len(seq))
    return max_len


X_train, y_train = load_UCR_UEA_dataset(DATASET_NAME, split="train")
X_test, y_test = load_UCR_UEA_dataset(DATASET_NAME, split="test")

overall_max_length = max(max_series_length(X_train), max_series_length(X_test))
padder = PaddingTransformer(pad_length=overall_max_length)

X_train = padder.fit_transform(X_train)
X_test = padder.transform(X_test)

X_train = convert_to(X_train, to_type="numpy3D").astype(np.float32)
X_test = convert_to(X_test, to_type="numpy3D").astype(np.float32)

baseline_num_channels = X_train.shape[1]

print(f"Dataset: {DATASET_NAME}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Original number of channels: {baseline_num_channels}")
print(f"Number of classes: {len(np.unique(y_train))}")

Dataset: EthanolConcentration
Train shape: (261, 3, 1751)
Test shape:  (263, 3, 1751)
Original number of channels: 3
Number of classes: 4


In [44]:
start_time = pd.Timestamp.now()
baseline_transformer = MiniRocketMultivariate(
    num_kernels=NUM_KERNELS,
    random_state=RANDOM_STATE,
)
baseline_classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))

baseline_transformer.fit(X_train)
X_train_baseline_features = baseline_transformer.transform(X_train)
X_test_baseline_features = baseline_transformer.transform(X_test)

baseline_classifier.fit(X_train_baseline_features, y_train)
baseline_train_pred = baseline_classifier.predict(X_train_baseline_features)
baseline_test_pred = baseline_classifier.predict(X_test_baseline_features)

baseline_train_acc = 100 * accuracy_score(y_train, baseline_train_pred)
baseline_test_acc = 100 * accuracy_score(y_test, baseline_test_pred)

print(f"Baseline MiniRocket train accuracy: {baseline_train_acc:.3f}")
print(f"Baseline MiniRocket test accuracy:  {baseline_test_acc:.3f}")
end_time = pd.Timestamp.now()
baseline_time = end_time - start_time
# Time to seconds with 2 decimal places
baseline_time = baseline_time.total_seconds()
print(f"Elapsed time: {baseline_time:.2f} seconds")

Baseline MiniRocket train accuracy: 88.123
Baseline MiniRocket test accuracy:  31.939
Elapsed time: 9.71 seconds


In [45]:
start_time = pd.Timestamp.now()
roman = RomanOperator(
    S=4,
    alpha=0.5,
    min_timesteps_per_channel=32,
    normalization=True,
)

X_train_roman = roman.fit_transform(X_train)
X_test_roman = roman.transform(X_test)

roman_num_channels = X_train_roman.shape[1]

print(f"Selected scales: {roman.S_}")
print(f"Scale lengths: {roman.lengths_}")
print(f"Windows per scale: {roman.windows_}")
print(f"Common output length: {roman.L_base_}")
print(f"ROMAN pseudochannels: {roman_num_channels}")

roman_transformer = MiniRocketMultivariate(
    num_kernels=NUM_KERNELS,
    random_state=RANDOM_STATE,
)
roman_classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))

roman_transformer.fit(X_train_roman)
X_train_roman_features = roman_transformer.transform(X_train_roman)
X_test_roman_features = roman_transformer.transform(X_test_roman)

roman_classifier.fit(X_train_roman_features, y_train)
roman_train_pred = roman_classifier.predict(X_train_roman_features)
roman_test_pred = roman_classifier.predict(X_test_roman_features)

roman_train_acc = 100 * accuracy_score(y_train, roman_train_pred)
roman_test_acc = 100 * accuracy_score(y_test, roman_test_pred)

print(f" ")

print(f"ROMAN + MiniRocket train accuracy: {roman_train_acc:.3f}")
print(f"ROMAN + MiniRocket test accuracy:  {roman_test_acc:.3f}")
end_time = pd.Timestamp.now()
roman_time = end_time - start_time
# Time to seconds with 2 decimal places
roman_time = roman_time.total_seconds()
print(f"Elapsed time: {roman_time:.2f} seconds")

Selected scales: 4
Scale lengths: [1751, 876, 438, 219]
Windows per scale: [15, 7, 3, 1]
Common output length: 219
ROMAN pseudochannels: 78
 
ROMAN + MiniRocket train accuracy: 94.636
ROMAN + MiniRocket test accuracy:  38.023
Elapsed time: 4.12 seconds


In [46]:
comparison = pd.DataFrame(
    [
        {
            "Model": "MiniRocket baseline",
            "Input channels": baseline_num_channels,
            "Train accuracy": baseline_train_acc,
            "Test accuracy": baseline_test_acc,
            "Elapsed time": baseline_time,
        },
        {
            "Model": "ROMAN (S=4) + MiniRocket",
            "Input channels": roman_num_channels,
            "Train accuracy": roman_train_acc,
            "Test accuracy": roman_test_acc,
            "Elapsed time": roman_time,
        },
    ]
)
# Display table with 2 decimal places for accuracy and time
comparison.style.format({
    "Train accuracy": "{:.1f} %",
    "Test accuracy": "{:.1f} %",
    "Elapsed time": "{:.1f} s"
})

,Model,Input channels,Train accuracy,Test accuracy,Elapsed time
0,MiniRocket baseline,3,88.1 %,31.9 %,9.7 s
1,ROMAN (S=4) + MiniRocket,78,94.6 %,38.0 %,4.1 s
